## Repository path configuration
All repository files are accessed through relative paths. Run the notebook from the repository root or from the `notebooks/` directory.


In [ ]:
from pathlib import Path
import os

# Resolve the repository root whether Jupyter starts in the repository root
# or directly inside the notebooks directory.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)

DATA_DIR = ROOT / "data"
EXTERNAL_DATA_DIR = DATA_DIR / "external"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
OUTPUTS_DIR = ROOT / "outputs"
RESULTS_DIR = OUTPUTS_DIR / "results"
FIGURES_DIR = OUTPUTS_DIR / "figures"
EXTERNAL_MATERIALS_DIR = ROOT / "external_materials"
MODEL_WEIGHTS_DIR = EXTERNAL_MATERIALS_DIR / "model_weights"

for folder in [
    EXTERNAL_DATA_DIR,
    PROCESSED_DATA_DIR,
    PROCESSED_DATA_DIR / "logs",
    RESULTS_DIR,
    FIGURES_DIR,
    MODEL_WEIGHTS_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Repository root:", ROOT)


In [ ]:
# Google Drive mounting is intentionally not used in this repository.

In [ ]:
import os
os.environ["WANDB_API_KEY"] = os.getenv("WANDB_API_KEY", "")

In [ ]:
!pip install wandb -qU
import wandb
wandb.login(key=os.environ["WANDB_API_KEY"]) if os.getenv("WANDB_API_KEY") else None

In [ ]:
!pip install -U transformers
!pip install accelerate -U
!pip install transformers[torch]
!pip install bitsandbytes

In [ ]:
import pandas as pd
import numpy as np
import torch
import random
import time
import nltk
import string
import matplotlib.pyplot as plt
import seaborn as sns

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk.tag import pos_tag


In [ ]:
#  UNIVERSAL IMPORTS 
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Trainer,
    TrainingArguments,
    TrainerCallback
)

from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import bitsandbytes as bnb

In [ ]:
# NLTK
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

# Randomness
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
def get_wordnet_pos(word):
    tag = pos_tag([word])[0][1][0].upper()
    tag_dict = {"J": wordnet.ADJ, "N": wordnet.NOUN, "V": wordnet.VERB, "R": wordnet.ADV}
    return tag_dict.get(tag, wordnet.NOUN)

def clean_text(text):
    stop_words = set(stopwords.words("english"))
    lemmatizer = WordNetLemmatizer()
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    words = word_tokenize(text)
    filtered_words = [word for word in words if word not in stop_words]
    lemmatized_words = [lemmatizer.lemmatize(word, get_wordnet_pos(word)) for word in filtered_words]
    return " ".join(lemmatized_words)

In [ ]:
# Custom logging callback
class LossLoggerCallback(TrainerCallback):
    def __init__(self):
        self.records = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return

        epoch = logs.get("epoch", None)
        loss = logs.get("loss", None)
        eval_loss = logs.get("eval_loss", None)

        if epoch is not None and (loss is not None or eval_loss is not None):
            self.records.append({
                "epoch": epoch,
                "loss": loss,
                "eval_loss": eval_loss
            })

loss_logger = LossLoggerCallback()

In [ ]:
# Dataset class
class UniversalSeq2SeqDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    # για T5-base
    """def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels['input_ids'][idx])
        return item"""

    # για T5-large
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}

        labels = torch.tensor(self.labels['input_ids'][idx])
        labels[labels == tokenizer.pad_token_id] = -100
        item['labels'] = labels
        return item

    def __len__(self):
        return len(self.labels['input_ids'])

In [ ]:
# ============================================
#  LOAD DATASET 
# ============================================

df = pd.read_csv("data/external/sentences_dataset_45269.csv")

"""print("\nΠροτάσεις πριν τον καθαρισμό!\n")
print(df.head(20))

df["sentence"] = df["sentence"].apply(clean_text)

print("\nΠροτάσεις μετά τον καθαρισμό!\n")
print(df.head(20))"""

num_duplicates = df['sentence'].duplicated().sum()
print("Πλήθος διπλών εγγραφών:", num_duplicates)

before = len(df)
df = df.drop_duplicates(subset='sentence', keep='first')
after = len(df)

print("Διαγράφηκαν:", before - after)

polarity_stats = df['polarity'].value_counts()
print("Polarity Statistics:\n", polarity_stats)

train_data, temp_data = train_test_split(df, test_size=0.3, stratify=df['polarity'], random_state=SEED)
val_data, test_data = train_test_split(temp_data, test_size=0.5, stratify=temp_data['polarity'], random_state=SEED)

label_map = {0: "negative", 1: "neutral", 2: "positive"}
inverse_label_map = {"negative": 0, "neutral": 1, "positive": 2}

def format_data_for_t5(texts, labels):
    input_texts = ["classify sentiment: " + text for text in texts]
    output_texts = [label_map[label] for label in labels]
    return input_texts, output_texts

train_texts, train_labels = format_data_for_t5(train_data["sentence"].tolist(), train_data["polarity"].tolist())
val_texts, val_labels = format_data_for_t5(val_data["sentence"].tolist(), val_data["polarity"].tolist())
test_texts, test_labels = format_data_for_t5(test_data["sentence"].tolist(), test_data["polarity"].tolist())


In [ ]:
# ============================================
#  MODEL SELECTION 
# ============================================

print("\nΔιάλεξε μοντέλο:\n")
print("1 = t5-large")               # Αυτό χρησιμοποιείται μόνο.
print("2 = facebook/bart-base")
print("3 = google/pegasus-large")
print("4 = google/ul2-base")

choice = input("\nΠληκτρολόγησε επιλογή: ")

if choice == "1":
    #MODEL_NAME = "t5-base"
    MODEL_NAME = "t5-large"
elif choice == "2":
    MODEL_NAME = "facebook/bart-base"
elif choice == "3":
    MODEL_NAME = "google/pegasus-large"
elif choice == "4":
    MODEL_NAME = "google/ul2-base"
else:
    print("Μη έγκυρη επιλογή → default: t5-base")
    MODEL_NAME = "t5-base"

print(f"\nΕπιλέχθηκε μοντέλο: {MODEL_NAME}\n")

EPOCHS = int(input("Δώσε αριθμό εποχών εκπαίδευσης (3–20): "))
while not (3 <= EPOCHS <= 20):
    EPOCHS = int(input("Δώσε αριθμό εποχών εκπαίδευσης (3–20): "))

In [ ]:
# ============================================
#  UNIVERSAL MODEL LOADER 
# ============================================

print(f"\nΦόρτωση μοντέλου: {MODEL_NAME}\n")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,

    # T5-base
    #dtype=torch.float16 if torch.cuda.is_available() else None
)

if "t5" in MODEL_NAME.lower():
    print("T5-family detected → prefix OK.")

elif "bart" in MODEL_NAME.lower():
    print("BART detected.")
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.eos_token_id = tokenizer.eos_token_id

elif "pegasus" in MODEL_NAME.lower():
    print("PEGASUS detected.")
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.eos_token_id = tokenizer.eos_token_id

elif "ul2" in MODEL_NAME.lower():
    print("UL2 detected → adjusting settings.")
    model.config.max_length = 256
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.eos_token_id = tokenizer.eos_token_id

print("Μοντέλο & tokenizer φορτώθηκαν επιτυχώς!\n")

In [ ]:
# ============================================
#  TOKENIZATION 
# ============================================

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=256)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=256)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=256)

train_labels_enc = tokenizer(train_labels, truncation=True, padding=True, max_length=3)  # 10 για Τ5-base
val_labels_enc = tokenizer(val_labels, truncation=True, padding=True, max_length=3)      # 10 για Τ5-base
test_labels_enc = tokenizer(test_labels, truncation=True, padding=True, max_length=3)    # 10 για Τ5-base

train_dataset = UniversalSeq2SeqDataset(train_encodings, train_labels_enc)
val_dataset = UniversalSeq2SeqDataset(val_encodings, val_labels_enc)
test_dataset = UniversalSeq2SeqDataset(test_encodings, test_labels_enc)

from transformers import EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir='./results',
    run_name="Sentiment_Analysis_Universal_Seq2Seq",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    logging_dir='./logs',
    warmup_ratio=0.1,
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    gradient_accumulation_steps=4,
    report_to="none",
    save_total_limit=2,
    #max_grad_norm=1.0,
    learning_rate=2e-5,

    # T5-base
    #bf16=True

    #T5-large
    #bf16=False,
    #fp16=True,

)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    callbacks=[loss_logger, EarlyStoppingCallback(early_stopping_patience=4)]
)

In [ ]:
# ============================================
#  TRAIN 
# ============================================

start_time = time.time()
print("Ξεκινά η εκπαίδευση...\n")

#if hasattr(model, "gradient_checkpointing_enable"):
#    model.gradient_checkpointing_enable()

trainer.train()

end_time = time.time()
minutes = int((end_time - start_time) // 60)
seconds = int((end_time - start_time) % 60)
print(f"\nΣυνολικός χρόνος: {minutes} λεπτά {seconds} δευτερόλεπτα\n")

In [ ]:
# ============================================
#  LOGS TO CSV 
# ============================================

records = loss_logger.records
df_logs = pd.DataFrame(records)

train_df = df_logs[df_logs["loss"].notna()][["epoch", "loss"]].rename(columns={"loss": "train_loss"})
eval_df = df_logs[df_logs["eval_loss"].notna()][["epoch", "eval_loss"]].rename(columns={"eval_loss": "val_loss"})

df_loss = pd.merge(train_df, eval_df, on="epoch", how="left")
df_loss = df_loss.sort_values("epoch").reset_index(drop=True)

NAME = MODEL_NAME.replace("/", "_")
csv_path = f"data/processed/logs/loss_{NAME}.csv"
df_loss.to_csv(csv_path, index=False)
print("Saved loss CSV:", csv_path)

In [ ]:
# ============================================
#  SAVE MODEL 
# ============================================

model_path = f"external_materials/model_weights/{NAME}/saved_model"
tokenizer_path = f"external_materials/model_weights/{NAME}/saved_tokenizer"

model.save_pretrained(model_path)
tokenizer.save_pretrained(tokenizer_path)

print(f"Το μοντέλο '{MODEL_NAME}' και ο tokenizer αποθηκεύτηκαν επιτυχώς!")

In [ ]:
# ============================================
#  EVALUATION 
# ============================================

def evaluate_model(model, tokenizer, test_texts, test_labels):
    model.eval()
    predictions = []

    for text in test_texts:
        input_ids = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=256).input_ids.to(model.device)
        output_ids = model.generate(input_ids)
        predicted_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        predictions.append(predicted_text)

    true_labels_numeric = [inverse_label_map[label] for label in test_labels]
    predicted_labels_numeric = [inverse_label_map.get(label, 1) for label in predictions]

    report = classification_report(true_labels_numeric, predicted_labels_numeric,
                                   target_names=['Negative', 'Neutral', 'Positive'])
    print(report)

    with open(f"outputs/results/classification_reports/{NAME}_classification_report.txt", "w", encoding="utf-8") as f:
        f.write(report)


    plt.rcParams.update({'font.size': 11})
    plt.figure(figsize=(6, 4))

    cm = confusion_matrix(true_labels_numeric, predicted_labels_numeric)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Negative', 'Neutral', 'Positive'],
                yticklabels=['Negative', 'Neutral', 'Positive'])
    plt.xlabel('Predicted')
    plt.ylabel('True')

    plt.tight_layout()

    plt.savefig(
        f'outputs/figures/fig/{NAME}_heatmap_SA_citation.pdf',
        format='pdf',
        bbox_inches='tight',
        dpi=300
    )
    plt.show()

evaluate_model(trainer.model, tokenizer, test_texts, test_labels)

In [ ]:
###################################